# RevalExo external-validation audit

Audit annotated walking segments and timestamp units. RevalExo is held out from training.

In [1]:
from pathlib import Path
import pandas as pd
import h5py
ROOT=Path('../data/raw/revalexo').resolve()
rows=[]
for folder in sorted(ROOT.glob('raw_full_part*/raw_full/Subject*')):
    group=folder.name.rsplit('_',1)[-1]
    ann=pd.read_csv(folder/'annotations.csv')
    walking=ann[ann['Task'].str.contains('level ground walking',case=False,na=False)]
    with h5py.File(folder/'mvn-analyze.hdf5','r') as f:
        base=f['mvn-analyze/xsens-motion-trackers']
        t=base['time_since_start_s'][:].ravel()
        dt=pd.Series(t).diff().median()
        fs=1/(dt/1e6)
        channels=sorted(base.keys())
    for _,w in walking.iterrows():
        rows.append({'subject':folder.name,'group':group,'task':w['Task'],'start_frame':w.get('Start_frame',pd.NA),'end_frame':w.get('End_frame',pd.NA),'start_time_s':w['Start_toa_s'],'end_time_s':w['End_toa_s'],'duration_s':float(w['End_toa_s']-w['Start_toa_s']),'sampling_hz':fs,'timestamp_unit':'microseconds','channels':';'.join(channels)})
audit=pd.DataFrame(rows)
display(audit.groupby('group').agg(subjects=('subject','nunique'),walking_segments=('subject','size'),median_duration_s=('duration_s','median'),median_sampling_hz=('sampling_hz','median')))
display(audit.head())

,subjects,walking_segments,median_duration_s,median_sampling_hz
group,,,,
HC,7,1358,0.566667,58.851223
SR,10,410,7.366667,58.962261
ST,10,844,2.683333,58.851223


,subject,group,task,start_frame,end_frame,start_time_s,end_time_s,duration_s,sampling_hz,timestamp_unit,channels
0,Subject01_HC,HC,Level ground walking,15601,15735,1.755769e+09,1.755769e+09,4.466667,58.851223,microseconds,acceleration;counter;free_acceleration;gyrosco...
1,Subject01_HC,HC,Level ground walking,16366,16545,1.755769e+09,1.755769e+09,6.000000,58.851223,microseconds,acceleration;counter;free_acceleration;gyrosco...
2,Subject01_HC,HC,Level ground walking,16659,16690,1.755769e+09,1.755769e+09,1.033333,58.851223,microseconds,acceleration;counter;free_acceleration;gyrosco...
3,Subject01_HC,HC,Level ground walking,16900,16937,1.755769e+09,1.755769e+09,1.266667,58.851223,microseconds,acceleration;counter;free_acceleration;gyrosco...
4,Subject01_HC,HC,Level ground walking,17106,17212,1.755769e+09,1.755769e+09,3.533333,58.851223,microseconds,acceleration;counter;free_acceleration;gyrosco...


In [2]:
out=Path('../data/processed/revalexo_external_validation_audit.csv')
out.parent.mkdir(parents=True,exist_ok=True)
audit.to_csv(out,index=False)
print(f'Wrote {len(audit)} walking segments to {out}')

Wrote 2612 walking segments to ..\data\processed\revalexo_external_validation_audit.csv
